In [1]:
import pandas as pd # For DataFrames, Series, and reading csv data in.
import seaborn as sns # Graphing, built ontop of MatPlot for ease-of-use and nicer diagrams.
import matplotlib.pyplot as plt # MatPlotLib for graphing data visually. Seaborn more likely to be used.
import numpy as np # For manipulating arrays and changing data into correct formats for certain libraries
from sklearn.model_selection import train_test_split
%matplotlib inline
from sklearn.preprocessing import StandardScaler  # For Normalization
import scikitplot # Confusion matrix plotting

In [2]:
df1=pd.read_csv('data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
#df1=df1.sample(n=100000,random_state=24)
df2 = pd.read_csv('data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
#df2=df2.sample(n=100000,random_state=24)
df3 = pd.read_csv('data/Friday-WorkingHours-Morning.pcap_ISCX.csv')
#df3=df3.sample(n=100000,random_state=24)
df4 = pd.read_csv('data/Monday-WorkingHours.pcap_ISCX.csv')
#df4=df4.sample(n=100000,random_state=24)
df5 = pd.read_csv('data/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv')
#df5=df5.sample(n=100000,random_state=24)
df6 = pd.read_csv('data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')
#df6=df6.sample(n=100000,random_state=24)
df7 = pd.read_csv('data/Tuesday-WorkingHours.pcap_ISCX.csv')
#df7=df7.sample(n=100000,random_state=24)
df8 = pd.read_csv('data/Wednesday-workingHours.pcap_ISCX.csv')
#df8=df8.sample(n=100000,random_state=24)


In [3]:
df=pd.concat([df1,df2,df3,df4,df5,df6,df7,df8],axis=0)
del df1,df2,df3,df4,df5,df6,df7,df8

In [4]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
#handling nan and inf values
def clean_dataset(df_cleaned):
    assert isinstance(df_cleaned, pd.DataFrame), "df needs to be a pd.DataFrame"
    df_cleaned.dropna(inplace=True) #eliminating rows or columns containing NaN values.
    indices_to_keep = ~df_cleaned.isin([np.nan, np.inf, -np.inf]).any(axis=1)
    return df_cleaned[indices_to_keep].astype(np.float64)


df_cleaned = df.copy()
#.reset_index() method resets the Pandas Dataframe indexes, for the rows. Useful to do after merging rows, as this messes up the indexes.
df_cleaned = df_cleaned.reset_index()
df_cleaned.drop('index', axis=1, inplace=True)
from sklearn.preprocessing import LabelEncoder
Le=LabelEncoder()
'''
df_cleaned['label'] = df_cleaned['label'].replace({'BENIGN': 0, 'DDoS': 1 ,'PortScan':2, 'Bot':3,'Infiltration':4,'Web Attack � Brute Force':5,'Web Attack � XSS':6,
                       ValueError: Found input variables with inconsistent numbers of samples: [2262300, 565576]                            'Web Attack � Sql Injection':7,'FTP-Patator':8,'SSH-Patator':9,'DoS slowloris':10,'DoS Slowhttptest':11,'DoS Hulk':12,'DoS GoldenEye':13,'Heartbleed':14})
'''
df_cleaned['label']=Le.fit_transform(df_cleaned['label'])
Le.classes_
#for inverse transform
#Le.inverse_transform([number])


array(['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk',
       'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed',
       'Infiltration', 'PortScan', 'SSH-Patator',
       'Web Attack � Brute Force', 'Web Attack � Sql Injection',
       'Web Attack � XSS'], dtype=object)

In [5]:
df_cleaned = clean_dataset(df_cleaned)


In [6]:
def Pre_process_data(df,col):
    '''
    input data-frame and coumn name
    operation fills the nan values with the minimum value in thier respective colum
    output returns the pre-processed dataframe
    '''
    print("Name of column with NaN:"+str(col))
    print(df[col].value_counts(dropna=False,normalize=True).head())
    df[col].replace(np.inf,-1,inplace=True)
    return df

In [7]:
def reduce_mem_usage(df):
    '''
    
   
    '''
    #print("Memory usage of properties dataframe is:", df.memory_usage().sum()/1024**2, "MB")
    for col in df.columns:
        if df[col].dtype != object: #exclude strings
            print("***************************************")
            print("Column:", col)
            print("dtype before:", df[col].dtype)
            # make variables for int, max and min
            IsInt = False
            mx = df[col].max
            mn = df[col].min
            # NA values should be handled prior to this
            if not np.isfinite(df[col].all()):
                df=Pre_process_data(df,col)
                
            asint = df[col].fillna(0).astype(np.int64)
            result = (df[col] - asint)
            result = result.sum()
            if result > -0.01 and result < 0.01:
                IsInt=True
            
            if IsInt:
                if mn() >= 0:
                    if mx() < 255:
                        df[col] = df[col].astype(np.uint8)
                    elif mx() < 65535:
                        df[col] = df[col].astype(np.uint16)
                    elif mx() < 4294967295:
                        df[col] = df[col].astype(np.uint32)
                    else:
                        df[col] = df[col].astype(np.uint64)
                else:
                    if mn() > np.iinfo(np.int8).min and mx() < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif mn() > np.iinfo(np.int16).min and mx() < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif mn() > np.iinfo(np.int32).min and mx() < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                    elif mn() > np.iinfo(np.int64).min and mx() < np.iinfo(np.int64).max:
                        df[col] = df[col].astype(np.int64)
            # make float datatypes 32bit
            else:
                df[col] = df[col].astype(np.float32)

          # print new column type
        print("dtype after:", df[col].dtype)
        print("*********************")
                
    #print("mem usage after completion")
    #mem_usg = df.memory_usage().sum / 1024 ** 2
   # print("final memory usage",mem_usg, "MB")
    return df

In [8]:
df_cleaned=reduce_mem_usage(df_cleaned) 

***************************************
Column: destination_port
dtype before: float64
dtype after: uint32
*********************
***************************************
Column: flow_duration
dtype before: float64
dtype after: int32
*********************
***************************************
Column: total_fwd_packets
dtype before: float64
dtype after: uint32
*********************
***************************************
Column: total_backward_packets
dtype before: float64
dtype after: uint32
*********************
***************************************
Column: total_length_of_fwd_packets
dtype before: float64
dtype after: uint32
*********************
***************************************
Column: total_length_of_bwd_packets
dtype before: float64
dtype after: uint32
*********************
***************************************
Column: fwd_packet_length_max
dtype before: float64
dtype after: uint16
*********************
***************************************
Column: fwd_packet_length_m

In [9]:
df_cleaned['label'].value_counts()

label
0     2271320
4      230124
10     158804
2      128025
3       10293
7        7935
11       5897
6        5796
5        5499
1        1956
12       1507
14        652
9          36
13         21
8          11
Name: count, dtype: int64

In [10]:
%matplotlib inline
X=df_cleaned.drop('label',axis=1)
y=df_cleaned['label']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

In [12]:
corr_features={'fwd_iat_std','flow_iat_min','packet_length_variance','idle_min','fwd_psh_flags', 'bwd_psh_flags',
       'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length','fin_flag_count',
       'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count',
       'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'down/up_ratio',
       'average_packet_size', 'avg_fwd_segment_size', 'avg_bwd_segment_size',
       'fwd_header_length.1', 'fwd_avg_bytes/bulk', 'fwd_avg_packets/bulk',
       'fwd_avg_bulk_rate', 'bwd_avg_bytes/bulk', 'bwd_avg_packets/bulk',
       'bwd_avg_bulk_rate', 'subflow_fwd_packets', 'subflow_fwd_bytes',
       'subflow_bwd_packets', 'subflow_bwd_bytes', 'init_win_bytes_forward','init_win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
       'active_mean', 'active_std', 'active_max', 'active_min', 'idle_mean',
       'idle_std', 'idle_max', 'idle_min','destination_port','bwd_header_length'}

In [13]:
X_train.drop(corr_features,axis=1,inplace=True) #exporting does not preserve datatype hence increases size of file
X_test.drop(corr_features,axis=1,inplace=True)


In [14]:
sc=StandardScaler()
X_train.iloc[:,:]=sc.fit_transform(X_train.iloc[:,:]) #Must have equal len keys and value when setting with a ndarray
X_test.iloc[:,:]=sc.transform(X_test.iloc[:,:])

In [15]:
X_train=reduce_mem_usage(X_train)
X_test=reduce_mem_usage(X_test)


***************************************
Column: flow_duration
dtype before: int32
dtype after: int32
*********************
***************************************
Column: total_fwd_packets
dtype before: float64
dtype after: float32
*********************
***************************************
Column: total_backward_packets
dtype before: float64
dtype after: float32
*********************
***************************************
Column: total_length_of_fwd_packets
dtype before: float64
dtype after: float32
*********************
***************************************
Column: total_length_of_bwd_packets
dtype before: float64
dtype after: float32
*********************
***************************************
Column: fwd_packet_length_max
dtype before: float64
dtype after: float32
*********************
***************************************
Column: fwd_packet_length_min
dtype before: float64
dtype after: float32
*********************
***************************************
Column: fwd_packet

In [16]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(2262300, 33)
(2262300,)
(565576, 33)
(565576,)


In [17]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

In [18]:
nb=GaussianNB()
nb.fit(X_train,y_train)

GaussianNB()

In [19]:
y_test_p = nb.predict(X_test)

In [20]:
y_train_p = nb.predict(X_train)
from sklearn.metrics import accuracy_score

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_test_p)
train_accuracy = accuracy_score(y_train, y_train_p)

print("Training Accuracy:", train_accuracy * 100)
print("Test Accuracy:", test_accuracy * 100)
#saving model Using sklearn Joblib
import joblib  #joblib is preferred method

joblib.dump(nb, 'Naive_Bayes_model')

Training Accuracy: 79.65610219687929
Test Accuracy: 79.64588313506938


['Naive_Bayes_model']

In [ ]:
#SVM
from sklearn.svm import SVC
svc_model=SVC(kernel='poly', C=1, decision_function_shape='ovr')
svc_model.fit(X_train,y_train)


In [ ]:
y_test_p = svc_model.predict(X_test)

In [ ]:
y_train_p = svc_model.predict(X_train)

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_test_p)
train_accuracy = accuracy_score(y_train, y_train_p)

print("Training Accuracy:", train_accuracy * 100)
print("Test Accuracy:", test_accuracy * 100)

In [ ]:
joblib.dump(svc_model, 'SVC_model')